# Backtesting

This notebook is a scaffold for single-asset strategy backtesting workflows in the Pricing folder. It uses local notebook parameters, builds a simple moving-average regime signal, adds an inverse-volatility sizing strategy, and compares both approaches to buy-and-hold.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from Quantapp.data import yf as qa_yf
# Seed the local package import when the notebook starts in a subfolder.
for _project_root_candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_project_root_candidate / "Quantapp" / "project.py").exists():
        if str(_project_root_candidate) not in sys.path:
            sys.path.insert(0, str(_project_root_candidate))
        break
else:
    raise RuntimeError("Could not locate the project root containing Quantapp.")

from Quantapp.project import ensure_project_root_on_path

PROJECT_ROOT = ensure_project_root_on_path()


from Quantapp.visualization.views.single_asset_profile.pricing.backtesting import plot_backtest_equity_curves_view

warnings.filterwarnings("ignore")

In [ ]:
# Block 2: set notebook parameters

TIMEFRAME_PROFILES = {
    "swing": {"short": 3, "mid": 9, "long": 21},
    "position": {"short": 21, "mid": 50, "long": 200},
    "structural": {"short": 200, "mid": 500, "long": 1000},
}


def resolve_time_frame_map(strategy: str) -> dict[str, int]:
    normalized_strategy = str(strategy).strip().lower()
    if normalized_strategy not in TIMEFRAME_PROFILES:
        raise ValueError(
            f"Invalid trading_strategy '{strategy}'. "
            f"Expected one of: {list(TIMEFRAME_PROFILES.keys())}"
        )
    return dict(TIMEFRAME_PROFILES[normalized_strategy])

single_asset_params = {
    "ticker_str": "SOXL",
    "interval": "1d",
    "period": "20y",
    "risk_free_ticker": "^IRX",
    "benchmark_tickers": ["SPY"],
    "trading_strategy": "position",
    "length_of_plots": 20,
    "var_position_value": None,
}

ticker_str = single_asset_params["ticker_str"]
interval = single_asset_params["interval"]
period = single_asset_params["period"]
risk_free_ticker = single_asset_params["risk_free_ticker"]
benchmark_tickers = list(single_asset_params["benchmark_tickers"])
trading_strategy = single_asset_params["trading_strategy"]
length_of_plots = single_asset_params["length_of_plots"]
var_position_value = single_asset_params["var_position_value"]
time_frame_map = resolve_time_frame_map(trading_strategy)

single_asset_params

In [ ]:
price_frame = qa_yf.Ticker(ticker_str).history(period=period, interval=interval).copy()
if price_frame.empty:
    raise ValueError(f"No price history returned for {ticker_str}.")

price_frame.index = pd.to_datetime(price_frame.index).tz_localize(None)
price_frame = price_frame.sort_index()
price_frame["daily_return"] = price_frame["Close"].pct_change().fillna(0.0)
price_frame["short_ma"] = price_frame["Close"].rolling(time_frame_map["short"]).mean()
price_frame["long_ma"] = price_frame["Close"].rolling(time_frame_map["long"]).mean()
price_frame["signal"] = (price_frame["short_ma"] > price_frame["long_ma"]).astype(float)
price_frame["position"] = price_frame["signal"].shift(1).fillna(0.0)
price_frame["strategy_return"] = price_frame["position"] * price_frame["daily_return"]

price_frame["rolling_vol_21d"] = price_frame["daily_return"].rolling(21).std() * np.sqrt(252)
price_frame["inverse_vol_raw"] = 1 / price_frame["rolling_vol_21d"].replace(0, np.nan)
price_frame["inverse_vol_scale"] = price_frame["inverse_vol_raw"] / price_frame["inverse_vol_raw"].expanding().mean()
price_frame["inverse_vol_position"] = price_frame["inverse_vol_scale"].clip(lower=0.0, upper=1.5).shift(1).fillna(0.0)
price_frame["inverse_vol_return"] = price_frame["inverse_vol_position"] * price_frame["daily_return"]

price_frame["buy_hold_index"] = (1 + price_frame["daily_return"]).cumprod()
price_frame["strategy_index"] = (1 + price_frame["strategy_return"]).cumprod()
price_frame["inverse_vol_index"] = (1 + price_frame["inverse_vol_return"]).cumprod()

backtest_summary = pd.DataFrame([
    {
        "series": "Buy and hold",
        "cumulativeReturnPct": round((price_frame["buy_hold_index"].iloc[-1] - 1) * 100, 2),
        "annualizedVolPct": round(price_frame["daily_return"].std() * np.sqrt(252) * 100, 2),
    },
    {
        "series": "MA crossover strategy",
        "cumulativeReturnPct": round((price_frame["strategy_index"].iloc[-1] - 1) * 100, 2),
        "annualizedVolPct": round(price_frame["strategy_return"].std() * np.sqrt(252) * 100, 2),
    },
    {
        "series": "Inverse volatility strategy",
        "cumulativeReturnPct": round((price_frame["inverse_vol_index"].iloc[-1] - 1) * 100, 2),
        "annualizedVolPct": round(price_frame["inverse_vol_return"].std() * np.sqrt(252) * 100, 2),
    },
])

backtest_summary

In [ ]:
fig = plot_backtest_equity_curves_view(price_frame, ticker_label=ticker_str)
fig.show(config={"responsive": True, "displaylogo": False})